# Embedding of evoked responses and TCA
Inspired by 'ripples' paper where time are used as features to investigate diversity of responses across age and cells

In [ ]:
import numpy as np
import os

session_paths = [
    'data_proc_ssd/jm/jm064/2025-11-13_s',
    'data_proc_ssd/jm/jm064/2025-11-14_s',
    'data_proc_ssd/jm/jm064/2025-11-15_s',
    'data_proc_ssd/jm/jm064/2025-11-16_s',
    'data_proc_ssd/jm/jm064/2025-11-17_s',
    'data_proc_ssd/jm/jm064/2025-11-18_s'
]

t2p_indices_path = 'data_proc_ssd/jm/jm064/track2p/plane0_suite2p_indices.npy'

st_analysis = False


all_resp_evoked = []
for session_path in session_paths:
    all_resp_evoked.append(np.load(os.path.join(session_path, 'resp_evoked', 'resp_evoked.npy'), allow_pickle=True).item())

n_features = all_resp_evoked[0]['resp_mean'].shape[1]
n_days = len(all_resp_evoked)
n_neurons = all_resp_evoked[0]['resp_mean'].shape[0]

In [ ]:
n_trials = 60
all_resp_mean = np.zeros((n_days, n_neurons, n_features))
all_resp = np.zeros((n_days, n_trials, n_neurons, n_features))


for i, resp_evoked in enumerate(all_resp_evoked):
    all_resp_mean[i] = resp_evoked['resp_mean']
    print(resp_evoked['resp'].shape)
    all_resp[i] = resp_evoked['resp']

# TODO: THE MATCHING DOESN'T MAKE SENSE...

# now plot some example neurons across days
import matplotlib.pyplot as plt
neuron_idxs = np.int64(np.linspace(0, n_neurons-1, 50))

for neuron_idx in neuron_idxs:
    fig, axs = plt.subplots(1, n_days, figsize=(15, 1))
    for day in range(n_days):
        axs[day].plot(all_resp_mean[day, neuron_idx])
        axs[day].set_title(f'Day {day+1}')
    plt.suptitle(f'Neuron {neuron_idx}')
    plt.show()

In [ ]:
print(all_resp.shape)
# now flatten the data for dimensionality reduction (neurons*trials*days x features)
all_resp_flat = all_resp.reshape(-1, n_features)
print(all_resp_flat.shape)
# now label rows by day


In [ ]:
# now import and run umap
import umap
from sklearn.decomposition import PCA

In [ ]:
def zscore_rows(X):
    return (X - X.mean(axis=1, keepdims=True)) / X.std(axis=1, keepdims=True)

In [ ]:
# flatten all_resp_mean along the days dimension and add labels (to color code the points by day)
data_mn = all_resp_mean.reshape(-1, n_features)
data_st = all_resp_flat
# # Compute the PSD: Instead of feeding raw time series into UMAP, feed it the Power Spectral Density (PSD) or the magnitude of the Fast Fourier Transform (FFT).
# data_psd = np.abs(np.fft.fft(data, axis=1))

# data = zscore_rows(data_psd)

labels_mn = np.repeat(np.arange(n_days), n_neurons)
labels_st = np.repeat(np.arange(n_days), n_neurons*n_trials)


In [ ]:
# now fit umap and visualise
reducer_mn = umap.UMAP(
    random_state=42,
    n_neighbors=15
)
emb_umap_mn = reducer_mn.fit_transform(data_mn)

In [ ]:
if st_analysis:
    reducer_st = umap.UMAP(
        random_state=42,
        n_neighbors=15
    )

    emb_umap_st = reducer_st.fit_transform(data_st)
else:
    print('ST analysis is disabled, skipping ST UMAP embedding.')

In [ ]:
nrn_idx = 306
nrn_idx_days = [nrn_idx + i*n_neurons for i in range(n_days)]

In [ ]:
plt.figure(figsize=(14, 10), dpi=300)
plt.scatter(emb_umap_mn[:, 0], emb_umap_mn[:, 1], s=30)
plt.axis('off')
cbar = plt.colorbar()
cbar.ax.tick_params(labelsize=18)
cbar.set_label('Postnatal day', fontsize=24)

In [ ]:
plt.figure(figsize=(14, 10), dpi=300)
plt.scatter(emb_umap_mn[:, 0], emb_umap_mn[:, 1], c=labels_mn+8, s=30, cmap='plasma')
plt.axis('off')
cbar = plt.colorbar()
cbar.ax.tick_params(labelsize=18)
cbar.set_label('Postnatal day', fontsize=24)

In [ ]:
if st_analysis:
    plt.figure(figsize=(14, 10), dpi=300)
    plt.scatter(emb_umap_st[:, 0], emb_umap_st[:, 1], c=labels_st+8, s=0.1, cmap='viridis', alpha=0.5)
    plt.axis('off')
    cbar = plt.colorbar()
    cbar.ax.tick_params(labelsize=18)
    cbar.set_label('Postnatal day', fontsize=24)
else:
    print('ST analysis is disabled, skipping ST UMAP plot.')

In [ ]:
# now get the centroid of the points corresponding to all of the trials of a given neuron on a given day

if st_analysis:
    nrn_idx = 33
    all_centroid = np.zeros((n_days, 2))
    for d in range(n_days):
        # get indices of the points corresponding to the trials of neuron nrn_idx on day d
        trial_indices = np.where(labels_st == d)[0]
        neuron_trial_indices = trial_indices[trial_indices % n_neurons == nrn_idx]
        neuron_trial_points = emb_umap_st[neuron_trial_indices]
        all_centroid[d] = neuron_trial_points.mean(axis=0)

        print(f'Centroid for neuron {nrn_idx} on day {d+1}: {all_centroid[d]}')
else:
    print('ST analysis is disabled, skipping centroid calculation.')

In [ ]:
ex_nrn_idxs = [33, 94, 284, 60, 47, 127, 282, 329]

In [ ]:
if st_analysis:
    for nrn_idx in ex_nrn_idxs:
        
        all_centroid = np.zeros((n_days, 2))
        for d in range(n_days):
            # get indices of the points corresponding to the trials of neuron nrn_idx on day d
            trial_indices = np.where(labels_st == d)[0]
            neuron_trial_indices = trial_indices[trial_indices % n_neurons == nrn_idx]
            neuron_trial_points = emb_umap_st[neuron_trial_indices]
            all_centroid[d] = neuron_trial_points.mean(axis=0)

            print(f'Centroid for neuron {nrn_idx} on day {d+1}: {all_centroid[d]}')

        plt.figure(figsize=(14, 10), dpi=300)
        plt.scatter(emb_umap_st[:, 0], emb_umap_st[:, 1], c=labels_st+8, s=0.1, cmap='viridis', alpha=0.5)
        # now plot the centroids
        plt.scatter(all_centroid[:, 0], all_centroid[:, 1], c=np.arange(n_days)+8, s=50, cmap='viridis', label=f'Neuron {nrn_idx} centroids', zorder=3)
        plt.plot(all_centroid[:, 0], all_centroid[:, 1], c='grey', linewidth=3, zorder=2, label='Trajectory')
        plt.legend()
        plt.axis('off')
        cbar = plt.colorbar()
        cbar.ax.tick_params(labelsize=18)
        cbar.set_label('Postnatal day', fontsize=24)
        plt.show()
else:
    print('ST analysis is disabled, skipping centroid plotting.')

In [ ]:
for nrn_idx in ex_nrn_idxs:
    nrn_idx_days = [nrn_idx + i*n_neurons for i in range(n_days)]
    
    fig, axs = plt.subplot_mosaic(mosaic='AAAAAA\nAAAAAA\nAAAAAA\nAAAAAA\nAAAAAA\nAAAAAA\nBCDEFG', figsize=(10, 10), dpi=300)
    # make the BCDEFG share y axis
    for ax in ['B', 'C', 'D', 'E', 'F', 'G']:
        axs[ax].sharey(axs['B'])
    axs['A'].scatter(emb_umap_mn[:, 0], emb_umap_mn[:, 1], c=labels_mn, s=5, cmap='plasma',alpha=0.7, zorder=0)
    axs['A'].scatter(emb_umap_mn[nrn_idx_days, 0], emb_umap_mn[nrn_idx_days, 1], c=np.arange(n_days), s=200, cmap='plasma')
    axs['A'].plot(emb_umap_mn[nrn_idx_days, 0], emb_umap_mn[nrn_idx_days, 1], c='gray', alpha=0.5, label=f'Trajectory of neuron {nrn_idx}', linewidth=5)
    axs['A'].set_title('UMAP embedding of evoked responses')
    axs['A'].set_xlabel('UMAP 1')
    axs['A'].set_ylabel('UMAP 2')
    # remove axis
    axs['A'].axis('off')
    # add legend to top left corner of the plot
    axs['A'].legend(loc='upper left', frameon=False)
    # add colormap labelled with the day numbers
    other_days = 'BCDEFG'
    for day in range(n_days):
        # get color based on 'plasma' colormap and the day index
        color = plt.cm.plasma(day/ (n_days-1))
        axs[other_days[day]].plot(all_resp_mean[day, nrn_idx], color=color, linewidth=3.5)
        axs[other_days[day]].set_xticks([])
        axs[other_days[day]].set_yticks([]) 
        axs[other_days[day]].set_axis_off()
        # add f'P{8+day}' to top left corner of the subplot
        axs[other_days[day]].text(0.05, 0.95, f'P{8+day}', transform=axs[other_days[day]].transAxes, fontsize=16, verticalalignment='top', color=color)


In [ ]:
import tensortools as tt

data = all_resp_mean # ... specify a numpy array holding the tensor you wish to fit
# data = all_resp.reshape(n_days, n_trials*n_neurons, n_features) # ... specify a numpy array holding the tensor you wish to fit

# Fit an ensemble of models, 4 random replicates / optimization runs per model rank
ensemble = tt.Ensemble(fit_method="ncp_hals")
ensemble.fit(data, ranks=range(1, 10), replicates=5)

fig, axes = plt.subplots(1, 2)
tt.plot_objective(ensemble, ax=axes[0])   # plot reconstruction error as a function of num components.
tt.plot_similarity(ensemble, ax=axes[1])  # plot model similarity as a function of num components.
fig.tight_layout()

# Plot the low-d factors for an example model, e.g. rank-2, first optimization run / replicate.
num_components = 4
replicate = 0
# color lines in 'C1' 
tt.plot_factors(ensemble.factors(num_components)[replicate], line_kw=[{'color': 'C2'}, {'color': 'C0'}, {'color': 'C3'}])  # plot the low-d factors

plt.show()

In [ ]:
day_components = np.zeros((num_components, n_days))
nrn_components = np.zeros((num_components, n_neurons))
feature_components = np.zeros((num_components, n_features))

for i in range(num_components):
    day_components[i] = ensemble.factors(num_components)[replicate][0][:, i]
    nrn_components[i] = ensemble.factors(num_components)[replicate][1][:, i]
    feature_components[i] = ensemble.factors(num_components)[replicate][2][:, i]

In [ ]:
fig, axs = plt.subplots(num_components, 1, figsize=(2, 5))
for i in range(num_components):
    axs[i].hist(nrn_components[i,:], bins=26)
    axs[i].axis('off')
plt.show()

In [ ]:
import matplotlib.colors as colors


In [ ]:
# now scatter the umap embedding color coded by the TCA component embedding
for i in range(num_components):
    # outer product of neuron and day componentsa
    c = np.outer(day_components[i], nrn_components[i, :]).flatten()
    # now do the log to get a logarithmic color scale (since the values are mostly close to zero, with some large outliers)
    c = np.log(np.abs(c) + 1e-5)  # add a small value to avoid log(0)
    
    plt.figure(figsize=(10, 8))
    plt.scatter(
        emb_umap_mn[:, 0],
        emb_umap_mn[:, 1],
        c=c,
        s=5,
        cmap='viridis'
    )
    plt.axis('off')
    cbar = plt.colorbar(ticks=[])
    cbar.set_label(fr'$TC{i+1}_{{\mathrm{{day}}}}\otimes TC{i+1}_{{\mathrm{{neuron}}}}$ (log scale)', fontsize=12)
    plt.title(f'UMAP embedding colored by TCA component {i+1}')
    plt.xlabel('UMAP 1')
    plt.ylabel('UMAP 2')
    plt.show()

In [ ]:
# now scatter the umap embedding color coded by the TCA component embedding
for i in range(num_components):
    # outer product of neuron and day componentsa
    c = np.outer(day_components[i], nrn_components[i, :]).flatten()
    # now do the log to get a logarithmic color scale (since the values are mostly close to zero, with some large outliers)
    c = np.log(np.abs(c) + 1e-5)  # add a small value to avoid log(0)
    
    fig, axs = plt.subplot_mosaic(mosaic='AAA\nAAA\nAAA\nAAA\nAAA\nAAA\nBCD', figsize=(10, 8), dpi=300)
    axs['A'].scatter(
        emb_umap_mn[:, 0],
        emb_umap_mn[:, 1],
        c=c,
        s=5,
        cmap='viridis'
    )
    axs['A'].axis('off')
    cbar = fig.colorbar(axs['A'].collections[0], ax=axs['A'], ticks=[], shrink=0.8)
    cbar.set_label(fr'$TC{i+1}_{{\mathrm{{day}}}}\otimes TC{i+1}_{{\mathrm{{neuron}}}}$ (log scale)', fontsize=12)
    axs['A'].set_title(f'UMAP embedding colored by TCA component {i+1}')
    axs['B'].plot(day_components[i], color='C2')
    axs['B'].set_title(f'Day component {i+1}')
    axs['B'].set_axis_off()
    axs['C'].plot(nrn_components[i, :], color='C0')
    axs['C'].set_title(f'Neuron component {i+1}')
    axs['C'].set_axis_off()
    axs['D'].plot(feature_components[i, :], color='C3')
    axs['D'].set_title(f'Feature component {i+1}')
    axs['D'].set_axis_off()

    plt.show()

In [ ]:
# now do a raster plot just showing the dynamics of each day side by side

## 1 · PSTH heatmaps (representation-drift style)

In [ ]:
# ─────────────────────────────────────────────────────────────────
# 1 · PSTH heatmaps – representation-drift style
#
# Three sortings (rows fixed across days within each figure):
#   (a) default order
#   (b) sorted by stimulus response on the FIRST day
#   (c) sorted by stimulus response on the LAST day
#
# Style follows the reference figure: narrow greyscale panels,
# minimal axes, day labels only on the first and last panel.
# ─────────────────────────────────────────────────────────────────

STIM_START  = 15
STIM_FRAMES = 60
STIM_END    = STIM_START + STIM_FRAMES   # = 75
CMAP_PSTH   = 'Greys'     # greyscale, high activity = dark
FIG_DPI     = 400

# Width of each day panel in inches; height scaled to neuron count
PANEL_W = 1.1
PANEL_H = 4.0


def psth_heatmap(all_resp_mean, neuron_order, sort_label,
                 stim_start=STIM_START, stim_end=STIM_END):
    """Plot PSTH heatmaps for every day in the representation-drift style.

    Each day is a narrow panel; rows are neurons in a fixed order;
    columns are time-frames.  Stimulus onset/offset are marked with
    thin vertical lines.  Only the first and last day get P-day labels.

    Parameters
    ----------
    all_resp_mean : ndarray (n_days, n_neurons, n_features)
    neuron_order  : 1-D int array   – fixed row ordering across all days
    sort_label    : str             – used in the y-axis label and title
    stim_start    : int
    stim_end      : int
    """
    n_days, n_neurons, n_features = all_resp_mean.shape

    # clip to ±2 SD for contrast, then map to [0, 1] for Greys
    flat  = all_resp_mean[:, neuron_order, :]
    vmax  = 4 * np.std(flat)
    vmin  = 0

    fig, axs = plt.subplots(
        1, n_days,
        figsize=(PANEL_W/2 * n_days, PANEL_H/2),
        dpi=FIG_DPI,
    )
    fig.subplots_adjust(left=0.08, right=0.97, top=0.88,
                        bottom=0.06, wspace=0.04)
    if n_days == 1:
        axs = [axs]

    for d, ax in enumerate(axs):
        img = all_resp_mean[d, neuron_order, :]

        ax.imshow(
            img,
            aspect='auto',
            cmap=CMAP_PSTH,
            vmin=vmin,
            vmax=vmax,
            interpolation='nearest',
        )

        # stimulus onset / offset lines
        for xpos in (stim_start - 0.5, stim_end - 0.5):
            ax.axvline(xpos, color='C3', lw=0.7, alpha=0.3, linestyle='--')

        # remove all ticks and spines
        ax.set_xticks([])
        ax.set_yticks([])

        # P-day label above first and last panel only
        if d == 0:
            ax.set_title(f'P{8 + d}', fontsize=9, pad=3)
        elif d == n_days - 1:
            ax.set_title(f'P{8 + d}', fontsize=9, pad=3)

    # y-axis label on leftmost panel (rotated, outside)
    axs[0].set_ylabel('Cells', fontsize=8, rotation=90, labelpad=3)
    axs[0].set_yticks([0, n_neurons - 1])
    axs[0].set_yticklabels(['0', str(n_neurons)], fontsize=7)

    # x-axis ticks only on leftmost panel (onset and offset frames)
    axs[0].set_xticks([stim_start, stim_start + (stim_end - stim_start) // 2, stim_end])
    axs[0].set_xticklabels([0, 1, 2], fontsize=6)
    axs[0].set_xlabel('Time (s)', fontsize=8, labelpad=3)

    fig.suptitle(sort_label)
    plt.show()


# (a) default ordering
psth_heatmap(all_resp_mean, np.arange(n_neurons), 'Default sorting')

# (b) sort by mean stimulus response on the FIRST day
sort_first = np.argsort(
    all_resp_mean[0, :, STIM_START:STIM_END].mean(axis=1)
)[::-1]
psth_heatmap(all_resp_mean, sort_first, 'First day sorting')

# (c) sort by mean stimulus response on the LAST day
sort_last = np.argsort(
    all_resp_mean[-1, :, STIM_START:STIM_END].mean(axis=1)
)[::-1]
psth_heatmap(all_resp_mean, sort_last, 'Last day sorting')


## 2 · Manual-feature correlations

In [ ]:
# ─────────────────────────────────────────────────────────────────
# 2 · Manual feature definitions & correlation with neural responses
# ─────────────────────────────────────────────────────────────────

STIM_START   = 15
STIM_FRAMES  = 60
STIM_END     = STIM_START + STIM_FRAMES   # = 75
AGE_OFFSET   = 8


# ── 2a · Feature templates ───────────────────────────────────────

def make_manual_features(n_features, stim_start=STIM_START, stim_frames=STIM_FRAMES):
    """Return a dict of named unit-norm, zero-mean feature templates.

    All features are non-zero only inside the stimulus window and are
    mean-subtracted so that cells with a flat sustained response do not
    project onto the ramping filters.

    Features
    --------
    on-off  : square pulse (1 during stim, 0 elsewhere) — NOT mean-subtracted
              because it is intended to capture plateau responses
    sin osc : 5 full sine cycles during stimulus (zero mean by construction)
    ramp    : linear ramp 0→1 with mean removed  → captures ramp-up only
    """
    stim_end = stim_start + stim_frames
    t_stim   = np.arange(stim_frames, dtype=float)

    def _embed(kernel):
        """Place a stim-length kernel at the correct position in the full vector."""
        f = np.zeros(n_features)
        f[stim_start:stim_end] = kernel
        return f

    on_off  = _embed(np.ones(stim_frames))
    on_off /= np.max(np.abs(on_off))  # scale to ±1

    # 5 full periods; full-cycle sin already has zero mean
    sin_osc = _embed(np.sin(2 * np.pi * 5 * t_stim / stim_frames))
    sin_osc /= np.max(np.abs(sin_osc))  # scale to ±1

    # ramp up: subtract mean so a flat plateau gives r ≈ 0
    ramp_raw = np.linspace(0, 1, stim_frames)
    ramp     = _embed(ramp_raw - ramp_raw.mean())
    ramp /= np.max(np.abs(ramp))  # scale to ±1

    raw = {'on-off': on_off, 'sin osc': sin_osc, 'ramp': ramp}
    return {k: v / (np.linalg.norm(v) + 1e-12) for k, v in raw.items()}


features = make_manual_features(n_features)


# ── 2b · Plot feature templates ──────────────────────────────────

fig, axs = plt.subplots(1, len(features), figsize=(3 * len(features), 2.5), dpi=FIG_DPI)
feature_colors = ['C0', 'C1', 'C2']

for ax, (name, f), color in zip(axs, features.items(), feature_colors):
    f /= np.max(np.abs(f))  # ensure ±1 range for plotting
    ax.plot(f, color=color, lw=2)
    ax.axvspan(STIM_START, STIM_END, color='lightgray', alpha=0.3)
    ax.axhline(0, color='k', lw=0.5, ls='--')
    ax.set_title(name, fontsize=10)
    ax.set_xlabel('Time (frames)', fontsize=8)
    ax.set_ylabel('Amplitude', fontsize=8)
    ax.tick_params(labelsize=7)
    ax.spines[['top', 'right']].set_visible(False)

fig.suptitle('Manual feature templates', fontsize=12)
plt.tight_layout()
plt.show()


# ── 2c · Zero-lag Pearson correlation ────────────────────────────

def compute_feature_correlations(all_resp_mean, features):
    """Pearson r at zero lag between each neuron-day PSTH and each template.

    Parameters
    ----------
    all_resp_mean : ndarray (n_days, n_neurons, n_features)
    features      : dict {name: 1-D array length n_features}

    Returns
    -------
    corr_dict  : dict {name: ndarray (n_days * n_neurons,)}
    age_labels : 1-D int array
    """
    n_days, n_neurons, nf = all_resp_mean.shape
    X  = all_resp_mean.reshape(-1, nf)
    Xz = X - X.mean(axis=1, keepdims=True)
    sd = Xz.std(axis=1, keepdims=True)
    sd[sd == 0] = 1.0
    Xz /= sd

    corr_dict = {}
    for name, f in features.items():
        fz = (f - f.mean()) / (f.std() + 1e-12)
        corr_dict[name] = (Xz * fz).mean(axis=1)

    age_labels = np.repeat(np.arange(n_days) + AGE_OFFSET, n_neurons)
    return corr_dict, age_labels


corr_dict, age_labels = compute_feature_correlations(all_resp_mean, features)


# ── 2d · Scatter matrix helper ───────────────────────────────────

def plot_scatter_grid(corr_dict, age_labels, title, xlim=(-1, 1), ylim=(-1, 1)):
    """Scatter matrix of feature values, colour = postnatal day.

    Diagonal: per-day histograms.  Off-diagonal: scatter of feature pairs.
    Axes are fixed to xlim / ylim for cross-plot comparability.

    Parameters
    ----------
    corr_dict  : dict {feature_name: 1-D float array}
    age_labels : 1-D int array
    title      : str
    xlim, ylim : tuple  – fixed axis limits (default −1 to 1)
    """
    feat_names  = list(corr_dict.keys())
    n_feats     = len(feat_names)
    unique_ages = np.unique(age_labels)
    n_u         = len(unique_ages)

    fig, axs = plt.subplots(n_feats, n_feats,
                            figsize=(2.8 * n_feats, 2.8 * n_feats), dpi=FIG_DPI)
    if n_feats == 1:
        axs = np.array([[axs]])

    for r in range(n_feats):
        for c in range(n_feats):
            ax = axs[r, c]
            if r == c:
                for di, age in enumerate(unique_ages):
                    idx   = np.where(age_labels == age)[0]
                    color = plt.cm.plasma(di / max(n_u - 1, 1))
                    ax.hist(corr_dict[feat_names[r]][idx], bins=20, color=color,
                            alpha=0.5, density=True, histtype='stepfilled',
                            range=xlim)
                ax.set_xlim(xlim)
            else:
                ax.scatter(
                    corr_dict[feat_names[c]],
                    corr_dict[feat_names[r]],
                    c=age_labels, cmap='plasma',
                    s=2, alpha=0.5, rasterized=True,
                    vmin=age_labels.min(), vmax=age_labels.max(),
                )
                ax.set_xlim(xlim)
                ax.set_ylim(ylim)
                ax.set_xlabel(feat_names[c], fontsize=7)
                ax.set_ylabel(feat_names[r], fontsize=7)

            ax.axhline(0, lw=0.4, color='k', ls='--')
            ax.axvline(0, lw=0.4, color='k', ls='--')
            ax.tick_params(labelsize=6)
            ax.spines[['top', 'right']].set_visible(False)

    sm = plt.cm.ScalarMappable(
        cmap='plasma',
        norm=plt.Normalize(vmin=age_labels.min(), vmax=age_labels.max()),
    )
    sm.set_array([])
    cbar = fig.colorbar(sm, ax=axs, shrink=0.4, pad=0.02)
    cbar.set_label('Postnatal day', fontsize=10)
    cbar.ax.tick_params(labelsize=8)
    fig.suptitle(title, fontsize=12)
    plt.show()


# ── 2e · Max cross-correlation (phase-invariant) ─────────────────

def compute_xcorr_features(all_resp_mean, features,
                            stim_start=STIM_START, stim_frames=STIM_FRAMES):
    """Phase-invariant feature matching via maximum cross-correlation.

    For each neuron-day response × template, the template is shifted across
    lags ±(stim_frames // 2) and the lag giving maximum Pearson r is recorded.
    Edge wrap-around is suppressed by zeroing out the rolled region.

    Parameters
    ----------
    all_resp_mean : ndarray (n_days, n_neurons, n_features)
    features      : dict {name: 1-D array length n_features}

    Returns
    -------
    xcorr_dict  : dict {name + ' (xcorr)': ndarray (n_days * n_neurons,)}
    lag_dict    : dict {name + ' lag': ndarray (n_days * n_neurons,)}  integer frames
    age_labels  : 1-D int array
    """
    n_days, n_neurons, nf = all_resp_mean.shape
    N        = n_days * n_neurons
    max_lag  = stim_frames // 2
    lag_grid = np.arange(-max_lag, max_lag + 1)

    X  = all_resp_mean.reshape(-1, nf)
    Xz = X - X.mean(axis=1, keepdims=True)
    sd = Xz.std(axis=1, keepdims=True)
    sd[sd == 0] = 1.0
    Xz /= sd

    xcorr_dict = {}
    lag_dict   = {}

    for name, f in features.items():
        max_corr = np.full(N, -np.inf)
        best_lag = np.zeros(N, dtype=int)

        for lag in lag_grid:
            f_shifted = np.roll(f, lag)
            if lag > 0:
                f_shifted[:lag]  = 0.0
            elif lag < 0:
                f_shifted[lag:]  = 0.0

            fz = (f_shifted - f_shifted.mean()) / (f_shifted.std() + 1e-12)
            r  = (Xz * fz).mean(axis=1)

            improved           = r > max_corr
            max_corr[improved] = r[improved]
            best_lag[improved] = lag

        xcorr_dict[f'{name} (xcorr)'] = max_corr
        lag_dict[f'{name} lag']       = best_lag.astype(float)

    age_labels = np.repeat(np.arange(n_days) + AGE_OFFSET, n_neurons)
    return xcorr_dict, lag_dict, age_labels


xcorr_dict, lag_dict, age_labels_xc = compute_xcorr_features(all_resp_mean, features)


plot_scatter_grid(corr_dict, age_labels,
                  'Zero-lag feature correlations (colour = postnatal day)')

# xcorr scatter (correlations only, −1..1 range)
plot_scatter_grid(xcorr_dict, age_labels_xc,
                  'Max cross-correlation features (colour = postnatal day)', xlim=(-0.1, 1), ylim=(-0.1, 1))

# lag scatter (lag values; use full range rather than −1..1)
lag_max = STIM_FRAMES // 2
plot_scatter_grid(lag_dict, age_labels_xc,
                  'Best-lag per feature (frames; colour = postnatal day)',
                  xlim=(-lag_max, lag_max), ylim=(-lag_max, lag_max))


## 3 · NMF population dynamics

In [ ]:
# ─────────────────────────────────────────────────────────────────
# 3 · NMF population dynamics
#
# X (n_neurons × n_features) ≈ W H
#   W (n_neurons  × k) – neuron loadings
#   H (k × n_features) – temporal bases
#
# H.T gives one coordinate per time-frame; we embed those n_features
# points (= 90) into 2-D UMAP and plot the time-trajectory.
# Colour encodes the time-frame index (jet colormap).
# ─────────────────────────────────────────────────────────────────

from sklearn.decomposition import NMF
import umap
from scipy.interpolate import splprep, splev

NMF_COMPONENTS = 6
NMF_SEED       = 42


# ── 3a · Helper functions ────────────────────────────────────────

def fit_nmf(X, n_components=NMF_COMPONENTS, seed=NMF_SEED):
    """Fit NMF (ReLU pre-processing) to a PSTH matrix.

    Parameters
    ----------
    X : ndarray (n_neurons, n_features)

    Returns
    -------
    W     : ndarray (n_neurons,    n_components)
    H     : ndarray (n_components, n_features)
    model : fitted sklearn NMF
    """
    X_nn  = np.clip(X, 0, None)
    model = NMF(n_components=n_components, init='nndsvda',
                random_state=seed, max_iter=500)
    W = model.fit_transform(X_nn)
    return W, model.components_, model


def project_nmf(X, model):
    """Project data onto a pre-fitted NMF basis (returns W)."""
    return model.transform(np.clip(X, 0, None))


def get_time_coords(H):
    """Return time-point coordinates in NMF component space.

    Parameters
    ----------
    H : ndarray (n_components, n_features)

    Returns
    -------
    T : ndarray (n_features, n_components)  – one row per time-frame
    """
    return H.T


def fit_umap_on_coords(T_list, seed=NMF_SEED):
    """Jointly embed a list of (n_features, n_components) matrices in 2-D.

    Returns
    -------
    embeddings : list of ndarray (n_features, 2), one per input matrix
    reducer    : fitted umap.UMAP
    """
    T_cat   = np.vstack(T_list)
    reducer = umap.UMAP(n_neighbors=min(15, len(T_cat) - 1),
                        random_state=seed, min_dist=0.3)
    emb_all = reducer.fit_transform(T_cat)
    nf      = T_list[0].shape[0]
    return [emb_all[d * nf:(d + 1) * nf] for d in range(len(T_list))], reducer


def _interpolate_trajectory(xy, n_interp=300):
    """Fit a B-spline through the 2-D trajectory and return densely sampled points.

    Parameters
    ----------
    xy       : ndarray (n_points, 2)
    n_interp : int  – number of output points

    Returns
    -------
    xy_fine : ndarray (n_interp, 2)
    t_fine  : ndarray (n_interp,) in [0, 1]  – parametric position
    """
    # de-duplicate consecutive identical points (spline requires distinct knots)
    mask = np.concatenate(([True], np.any(np.diff(xy, axis=0) != 0, axis=1)))
    xy   = xy[mask]
    k    = min(3, len(xy) - 1)
    tck, u = splprep([xy[:, 0], xy[:, 1]], s=0, k=k)
    t_fine = np.linspace(0, 1, n_interp)
    x_fine, y_fine = splev(t_fine, tck)
    return np.column_stack([x_fine, y_fine]), t_fine


def _draw_coloured_line(ax, xy, t, cmap):
    """Draw a line whose colour varies along a 1-D parameter t ∈ [0, 1]."""
    from matplotlib.collections import LineCollection
    points  = xy.reshape(-1, 1, 2)
    segs    = np.concatenate([points[:-1], points[1:]], axis=1)
    lc      = LineCollection(segs, cmap=cmap,
                             norm=plt.Normalize(0, 1), lw=1.5, zorder=2)
    lc.set_array(t[:-1])
    ax.add_collection(lc)
    return lc


def plot_trajectories_per_day(embeddings, ages, title, interpolate=False,
                              stim_start=STIM_START, stim_end=STIM_END):
    """Plot each day's time-trajectory in its own axis, colour = time-frame (jet).

    Parameters
    ----------
    embeddings  : list of ndarray (n_features, 2)
    ages        : 1-D int array
    title       : str
    interpolate : bool – if True draw a smooth B-spline instead of raw points
    stim_start  : int
    stim_end    : int
    """
    n_days   = len(embeddings)
    n_frames = embeddings[0].shape[0]
    t_idx    = np.arange(n_frames)
    CMAP_T   = 'jet'

    fig, axs = plt.subplots(1, n_days, figsize=(3.2 * n_days, 3.5),
                            dpi=FIG_DPI, constrained_layout=True)
    if n_days == 1:
        axs = [axs]

    for d, (ax, emb, age) in enumerate(zip(axs, embeddings, ages)):
        day_color = plt.cm.plasma(d / max(n_days - 1, 1))

        if interpolate:
            xy_fine, t_fine = _interpolate_trajectory(emb)
            lc = _draw_coloured_line(ax, xy_fine, t_fine, CMAP_T)
            ax.autoscale()
        else:
            sc = ax.scatter(emb[:, 0], emb[:, 1],
                            c=t_idx, cmap=CMAP_T, s=14, zorder=3,
                            norm=plt.Normalize(0, n_frames - 1))
            ax.plot(emb[:, 0], emb[:, 1], color='grey', lw=0.5, alpha=0.4, zorder=2)

        # stim window in a contrasting highlight
        ax.scatter(emb[stim_start:stim_end, 0],
                   emb[stim_start:stim_end, 1],
                   s=20, color='tomato', zorder=4, linewidths=0, alpha=0.7)

        # onset (▲) and offset (▼) markers
        ax.scatter(*emb[stim_start],   s=70, color='k', marker='^', zorder=6)
        ax.scatter(*emb[stim_end - 1], s=70, color='k', marker='v', zorder=6)

        ax.set_title(f'P{age}', fontsize=10, color=day_color)
        ax.set_xlabel('NMF 1', fontsize=7)
        if d == 0:
            ax.set_ylabel('NMF 2', fontsize=7)
        ax.tick_params(labelsize=6)
        ax.spines[['top', 'right']].set_visible(False)

    # shared colourbar: time-frame
    sm = plt.cm.ScalarMappable(cmap=CMAP_T,
                               norm=plt.Normalize(0, n_frames - 1))
    sm.set_array([])
    cb = fig.colorbar(sm, ax=axs[-1], shrink=0.7, pad=0.03)
    cb.set_label('Time (frame)', fontsize=8)
    cb.ax.tick_params(labelsize=7)

    fig.suptitle(title, fontsize=11)
    plt.show()


def plot_trajectories_combined(embeddings, ages, title,
                               stim_start=STIM_START, stim_end=STIM_END):
    """Single axis, all days overlaid, colour = postnatal day (plasma)."""
    n_days = len(embeddings)
    norm   = plt.Normalize(vmin=ages.min(), vmax=ages.max())
    CMAP_A = plt.cm.plasma

    fig, ax = plt.subplots(figsize=(7, 5), dpi=FIG_DPI)

    for d, (emb, age) in enumerate(zip(embeddings, ages)):
        color = CMAP_A(norm(age))
        ax.plot(emb[:, 0], emb[:, 1], color=color, lw=1.5, alpha=0.85, zorder=2)
        ax.scatter(emb[:, 0], emb[:, 1], color=color, s=5, zorder=3)
        ax.scatter(*emb[stim_start], s=60, color=color,
                   marker='^', zorder=5, edgecolors='k', linewidths=0.5)
        ax.text(emb[stim_start, 0], emb[stim_start, 1],
                f' P{age}', fontsize=7, color=color, va='center')

    sm = plt.cm.ScalarMappable(cmap=CMAP_A, norm=norm)
    sm.set_array([])
    cbar = fig.colorbar(sm, ax=ax, shrink=0.8)
    cbar.set_label('Postnatal day', fontsize=10)
    cbar.ax.tick_params(labelsize=8)
    ax.set_title(title, fontsize=11)
    ax.set_xlabel('NMF dim 1', fontsize=9)
    ax.set_ylabel('NMF dim 2', fontsize=9)
    ax.spines[['top', 'right']].set_visible(False)
    ax.tick_params(labelsize=7)
    plt.tight_layout()
    plt.show()


ages = np.arange(n_days) + AGE_OFFSET


# ── 3b · Fit NMF independently per day ───────────────────────────

print('Fitting NMF independently per day …')
T_per_day, W_per_day, models_per_day = [], [], []
for d in range(n_days):
    W, H, model = fit_nmf(all_resp_mean[d])
    W_per_day.append(W)
    T_per_day.append(get_time_coords(H))
    models_per_day.append(model)
    print(f'  Day P{ages[d]}: reconstruction error = {model.reconstruction_err_:.4f}')

emb_indep, _ = fit_umap_on_coords(T_per_day)
plot_trajectories_per_day(emb_indep, ages,
    'NMF per day (independent) – raw', interpolate=False)
plot_trajectories_per_day(emb_indep, ages,
    'NMF per day (independent) – interpolated', interpolate=True)


# ── 3c · Fit on reference day, project all others ────────────────

for ref_label, ref_idx in [('first day (P8)', 0), ('last day (P13)', n_days - 1)]:
    print(f'\nFitting NMF on {ref_label}, projecting all others …')
    W_ref, H_ref, model_ref = fit_nmf(all_resp_mean[ref_idx])

    T_proj_list = []
    for d in range(n_days):
        W_proj  = project_nmf(all_resp_mean[d], model_ref)
        X_nn    = np.clip(all_resp_mean[d], 0, None)
        H_proj  = np.linalg.lstsq(W_proj, X_nn, rcond=None)[0]
        T_proj_list.append(get_time_coords(H_proj))

    emb_proj, _ = fit_umap_on_coords(T_proj_list)
    plot_trajectories_per_day(emb_proj, ages,
        f'NMF basis from {ref_label} – raw', interpolate=False)
    plot_trajectories_per_day(emb_proj, ages,
        f'NMF basis from {ref_label} – interpolated', interpolate=True)


# ── 3d · Concatenated NMF – all days, colour by age ──────────────

print('\nFitting NMF on concatenated PSTHs (all days) …')
X_cat               = np.vstack([all_resp_mean[d] for d in range(n_days)])
W_cat, _, model_cat = fit_nmf(X_cat)
print(f'  Global reconstruction error = {model_cat.reconstruction_err_:.4f}')

T_global_list = []
for d in range(n_days):
    W_d = W_cat[d * n_neurons:(d + 1) * n_neurons]
    X_d = np.clip(all_resp_mean[d], 0, None)
    H_d = np.linalg.lstsq(W_d, X_d, rcond=None)[0]
    T_global_list.append(get_time_coords(H_d))

emb_global, _ = fit_umap_on_coords(T_global_list)
plot_trajectories_combined(emb_global, ages,
    'NMF on concatenated PSTHs – trajectories colour-coded by age')


## 4 · UMAP coloured by feature correlations & optimal lags

Re-use the existing `emb_umap_mn` embedding (neurons × days flattened) and
colour each point by:
- zero-lag Pearson r with each template feature
- max-xcorr value for each feature
- optimal lag for each feature


In [ ]:
# ─────────────────────────────────────────────────────────────────
# 4 · UMAP colour-coded by feature correlations and optimal lags
#
# emb_umap_mn  shape (n_days * n_neurons, 2)  – existing embedding
# corr_dict    zero-lag correlations  (from §2c)
# xcorr_dict   max-xcorr correlations (from §2e)
# lag_dict     optimal lag per feature (from §2e)
# ─────────────────────────────────────────────────────────────────

# Colourmap choices: diverging for correlations, sequential for lags
CMAP_CORR = 'RdBu_r'   # −1 (blue) → 0 → +1 (red)
CMAP_LAG  = 'coolwarm'  # negative lag (blue) → 0 → positive lag (red)


def umap_coloured(emb, values, label, cmap, vmin, vmax, title):
    """Scatter the UMAP embedding coloured by a scalar field.

    Parameters
    ----------
    emb    : ndarray (N, 2)
    values : ndarray (N,)
    label  : str  – colourbar label
    cmap   : str or Colormap
    vmin, vmax : float  – colour scale limits
    title  : str
    """
    fig, ax = plt.subplots(figsize=(7, 5), dpi=FIG_DPI)
    sc = ax.scatter(emb[:, 0], emb[:, 1],
                    c=values, cmap=cmap,
                    vmin=vmin, vmax=vmax,
                    s=5, alpha=0.7, rasterized=True)
    cbar = fig.colorbar(sc, ax=ax, shrink=0.8, pad=0.02)
    cbar.set_label(label, fontsize=10)
    cbar.ax.tick_params(labelsize=8)
    ax.set_title(title, fontsize=11)
    ax.axis('off')
    plt.tight_layout()
    plt.show()


# ── 4a · Zero-lag correlations ───────────────────────────────────

for name, vals in corr_dict.items():
    umap_coloured(
        emb_umap_mn, vals,
        label=f'Pearson r  [{name}]',
        cmap=CMAP_CORR, vmin=-1, vmax=1,
        title=f'UMAP – zero-lag correlation with "{name}"',
    )

# ── 4b · Max cross-correlation values ────────────────────────────

for name, vals in xcorr_dict.items():
    umap_coloured(
        emb_umap_mn, vals,
        label=f'max xcorr  [{name}]',
        cmap=CMAP_CORR, vmin=-1, vmax=1,
        title=f'UMAP – max cross-correlation: "{name}"',
    )

# ── 4c · Optimal lag per feature ─────────────────────────────────

lag_max = STIM_FRAMES // 2

for name, vals in lag_dict.items():
    umap_coloured(
        emb_umap_mn, vals,
        label='Optimal lag (frames)',
        cmap=CMAP_LAG, vmin=-lag_max, vmax=lag_max,
        title=f'UMAP – optimal lag: "{name}"',
    )


## 5 · PCA population dynamics (parallel to NMF)

In [ ]:
# ─────────────────────────────────────────────────────────────────
# 5 · PCA population dynamics
#
# Same structure as the NMF section (§3) but using PCA instead.
# PCA does not require non-negativity and naturally orders components
# by explained variance.
#
# The score matrix S = X @ V.T  (n_neurons × n_components) gives
# neuron trajectories; V (n_components × n_features) gives temporal
# bases.  We embed V.T (n_features × n_components) — exactly
# analogous to NMF's H.T — to get the population time-trajectory.
# ─────────────────────────────────────────────────────────────────

from sklearn.decomposition import PCA

PCA_COMPONENTS = 6
PCA_SEED       = 42


# ── 5a · Helpers (mirror the NMF API) ────────────────────────────

def fit_pca(X, n_components=PCA_COMPONENTS):
    """Fit PCA on a PSTH matrix (no non-negativity constraint).

    Parameters
    ----------
    X : ndarray (n_neurons, n_features)

    Returns
    -------
    scores     : ndarray (n_neurons,   n_components)  – W analogue
    components : ndarray (n_components, n_features)   – H analogue
    model      : fitted sklearn PCA
    """
    model  = PCA(n_components=n_components, random_state=PCA_SEED)
    scores = model.fit_transform(X)
    return scores, model.components_, model


def project_pca(X, model):
    """Project new data onto a pre-fitted PCA basis."""
    return model.transform(X)


# fit_umap_on_coords, get_time_coords, _interpolate_trajectory,
# _draw_coloured_line, plot_trajectories_per_day, plot_trajectories_combined
# are all reused from §3 (already defined above).


# ── 5b · PCA independently per day ───────────────────────────────

print('Fitting PCA independently per day …')
T_pca_per_day = []
for d in range(n_days):
    scores, comps, pca_model = fit_pca(all_resp_mean[d])
    T_pca_per_day.append(get_time_coords(comps))   # (n_features, n_components)
    var = pca_model.explained_variance_ratio_
    print(f'  Day P{ages[d]}: cumulative var explained '
          f'({PCA_COMPONENTS} PCs) = {var.sum()*100:.1f}%')

emb_pca_indep, _ = fit_umap_on_coords(T_pca_per_day)
plot_trajectories_per_day(emb_pca_indep, ages,
    'PCA per day (independent) – raw', interpolate=False)
plot_trajectories_per_day(emb_pca_indep, ages,
    'PCA per day (independent) – interpolated', interpolate=True)


# ── 5c · Fit on reference day, project all others ────────────────

for ref_label, ref_idx in [('first day (P8)', 0), ('last day (P13)', n_days - 1)]:
    print(f'\nFitting PCA on {ref_label}, projecting all others …')
    _, comps_ref, pca_ref = fit_pca(all_resp_mean[ref_idx])

    T_pca_proj = []
    for d in range(n_days):
        scores_proj = project_pca(all_resp_mean[d], pca_ref)
        # reconstruct temporal coords via least-squares: X ≈ S_proj @ H_proj
        H_proj = np.linalg.lstsq(scores_proj, all_resp_mean[d], rcond=None)[0]
        T_pca_proj.append(get_time_coords(H_proj))

    emb_pca_proj, _ = fit_umap_on_coords(T_pca_proj)
    plot_trajectories_per_day(emb_pca_proj, ages,
        f'PCA basis from {ref_label} – raw', interpolate=False)
    plot_trajectories_per_day(emb_pca_proj, ages,
        f'PCA basis from {ref_label} – interpolated', interpolate=True)


# ── 5d · Concatenated PCA – all days, colour by age ──────────────

print('\nFitting PCA on concatenated PSTHs (all days) …')
X_cat_pca             = np.vstack([all_resp_mean[d] for d in range(n_days)])
_, _, pca_cat         = fit_pca(X_cat_pca)
W_cat_pca             = pca_cat.transform(X_cat_pca)
print(f'  Cumulative variance explained: '
      f'{pca_cat.explained_variance_ratio_.sum()*100:.1f}%')

T_pca_global = []
for d in range(n_days):
    W_d = W_cat_pca[d * n_neurons:(d + 1) * n_neurons]
    H_d = np.linalg.lstsq(W_d, all_resp_mean[d], rcond=None)[0]
    T_pca_global.append(get_time_coords(H_d))

emb_pca_global, _ = fit_umap_on_coords(T_pca_global)
plot_trajectories_combined(emb_pca_global, ages,
    'PCA on concatenated PSTHs – trajectories colour-coded by age')
